In [1]:
import numpy as np

class TradingEnv:
    def __init__(self, N=5, T=5.0, M=10, 
                 kappa=0.1, theta=10.0, sigma=0.01,
                 gamma=0.02, rho=0.5, eta=0.05,
                 b1=0.1, b2=0.5, b3=0.1):
        """
        Environnement de trading multi-agent (N agents) avec impact de marché.
        """
        self.N = N         # nombre d'agents
        self.T = T         # horizon de trading
        self.M = M         # nombre de pas discrets
        self.dt = T / M    # taille de pas

        # Dynamique du prix
        self.kappa = kappa  # vitesse de réversion
        self.theta = theta  # niveau moyen
        self.sigma = sigma  # volatilité

        # Impact de marché (transitoire/permanent)
        self.gamma = gamma  # amplitude de l'impact transitoire (sqrt)
        self.rho = rho      # taux de décroissance de l'impact transitoire
        self.eta = eta      # impact permanent linéaire

        # Coûts
        self.b1 = b1  # coût linéaire de transaction
        self.b2 = b2  # pénalité quadratique terminale
        self.b3 = b3  # pénalité de détention (risk penalty)

        self.reset()

    def reset(self):
        """
        Réinitialiser pour un nouvel épisode.
        """
        self.S = self.theta         # prix initial = long terme
        self.Y = 0.0                # aucun impact transitoire initial
        self.t = 0.0
        self.inv = np.ones(self.N)  # inventaire initial (1.0 chacun)
        self.inv0 = self.inv.copy()
        return self._get_state()

    def _get_state(self):
        """
        State vector = [prix normalisé, temps fractionnel, ordre cumulé, impact transitoire].
        """
        total_sold = np.sum(self.inv0 - self.inv)
        return np.array([
            self.S / self.theta,
            self.t / self.T,
            total_sold / np.sum(self.inv0),  # fraction de volume exécuté
            self.Y
        ], dtype=np.float32)

    def step(self, nu):
        """
        Prendre des actions de trading nu (taux de vente par agent).
        nu: array de shape (N,), nu[i] = taux de vente du ième agent (>0).
        Retourne: next_state, rewards, done.
        """
        nu = np.array(nu)
        r = np.zeros(self.N)
        terminal = (self.t >= self.T - 1e-9)

        # Calcul des récompenses avant mise à jour
        for i in range(self.N):
            # Coût de transaction linéaire + effet glissement
            r[i] = -nu[i] * (self.S + self.b1 * nu[i]) * self.dt
            # Pénalité d'inventaire courant (risk/holding penalty)
            r[i] -= self.b3 * (self.inv[i]**2) * self.dt
            # À l'étape terminale, liquidation
            if terminal:
                qT = self.inv[i] + nu[i]*self.dt
                r[i] += qT * (self.S - self.b2 * qT)

        # Mise à jour de l'inventaire et du temps
        self.inv = self.inv + nu * self.dt
        self.t += self.dt

        # Impact de marché transitoire
        nu_bar = np.sum(nu)
        Y_old = self.Y
        # Impact transitoire (sqrt rule)
        self.Y = (1 - self.rho * self.dt) * self.Y \
                 + self.gamma * np.sign(nu_bar) * np.sqrt(abs(nu_bar) + 1e-8) * self.dt
        dY = self.Y - Y_old

        # Dynamique du prix (Euler discretisation)
        drift = self.kappa * (self.theta - self.S) + self.eta * nu_bar
        noise = self.sigma * np.sqrt(self.dt) * np.random.randn()
        self.S = self.S + drift * self.dt + dY + noise

        # On empêche l'inventaire de devenir négatif
        self.inv = np.maximum(self.inv, 0.0)

        next_state = self._get_state()
        done = (self.t >= self.T - 1e-9)
        return next_state, r, done


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def mlp(sizes):
    """
    Construire un MLP séquentiel avec activations SiLU.
    """
    layers = []
    for i in range(len(sizes)-1):
        layers.append(nn.Linear(sizes[i], sizes[i+1]))
        if i < len(sizes)-2:
            layers.append(nn.SiLU())
    return nn.Sequential(*layers)

class PermInvariant(nn.Module):
    def __init__(self, n_agents):
        super().__init__()
        # phi: réseau partagé par agent sur l'inventaire (entrée 1)
        self.phi = mlp([1, 20, 20, 20])
        # rho: après sommation sur agents, on revient à 32-dim
        self.rho = mlp([20, 32])

    def forward(self, inv):
        # inv: shape (batch, n_agents)
        x = inv.unsqueeze(-1)      # (batch, n_agents, 1)
        x = self.phi(x)            # (batch, n_agents, 20)
        x = x.sum(dim=1)           # (batch, 20) sum sur les agents (invariance)
        return self.rho(x)         # (batch, 32)

class ValueNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Réseau de valeur: 4 → 32 → 32 → 32 → 32 → 1
        self.net = mlp([4, 32, 32, 32, 32, 1])

    def forward(self, x):
        return self.net(x)  # (batch, 4) → (batch, 1)

class AdvantageNet(nn.Module):
    def __init__(self, n_agents):
        super().__init__()
        self.n_agents = n_agents
        self.perm = PermInvariant(n_agents)
        # Main network: entrée = 4 états + 32 features invariantes = 36
        self.main = mlp([36, 32, 32, 32, 32])
        self.head = nn.Linear(32, 5)  # génère [μ, L, P12, P22, ψ]

    def forward(self, x, inv):
        # x: (batch, 4) état non-invar, inv: (batch, n_agents) inventaires
        B = x.shape[0]
        p = self.perm(inv)               # (batch, 32)
        h = torch.cat([x, p], dim=-1)    # (batch, 36)
        h = self.main(h)
        out = self.head(h)              # (batch, 5)
        mu = out[:, 0]
        L_raw = out[:, 1]
        P12 = out[:, 2]
        P22 = out[:, 3]
        psi = out[:, 4]
        # Assurer P11 >= 0 via Cholesky scaler
        P11 = L_raw.pow(2) + 1e-6

        # Étendre chaque paramètre (batch,1) → (batch,N)
        mu = mu.unsqueeze(1).expand(B, self.n_agents)
        P11 = P11.unsqueeze(1).expand(B, self.n_agents)
        P12 = P12.unsqueeze(1).expand(B, self.n_agents)
        P22 = P22.unsqueeze(1).expand(B, self.n_agents)
        psi = psi.unsqueeze(1).expand(B, self.n_agents)
        return mu, P11, P12, P22, psi

class NashDQN(nn.Module):
    def __init__(self, n_agents):
        super().__init__()
        self.n_agents = n_agents
        self.V = ValueNet()
        self.A = AdvantageNet(n_agents)

    def forward(self, state, inv, action):
        """
        Calcule Q(state, action) pour chaque agent et renvoie également psi (pour régularisation).
        state: (batch,4), inv: (batch,n), action: (batch,n)
        """
        # Valeur de base V(s) appliquée identique pour tous les agents
        V = self.V(state).expand(-1, self.n_agents)  # (batch, n_agents)
        # Paramètres de la fonction d'avantage
        mu, P11, P12, P22, psi = self.A(state, inv)
        # Calcul de l'avantage A_i comme dans Eq.(4.12)
        delta = action - mu
        total_delta = delta.sum(dim=1, keepdim=True)
        others = total_delta - delta
        A_val = (- P11 * delta**2
                 - P12 * delta * others
                 - P22 * others**2
                 + psi * others)
        return V + A_val, psi  # (batch, n_agents), (batch, n_agents)

    def nash_action(self, state, inv):
        """
        Calcule l'action en équilibre (u* = μ^θ(x)) pour état donné.
        """
        with torch.no_grad():
            mu, *_ = self.A(state, inv)
        return mu  # (batch, n_agents)


In [3]:
import torch.optim as optim
import random
from collections import deque

env = TradingEnv()
model = NashDQN(env.N)
target = NashDQN(env.N)
target.load_state_dict(model.state_dict())
optimizer_V = optim.Adam(model.V.parameters(), lr=3e-4)
optimizer_A = optim.Adam(model.A.parameters(), lr=3e-4)
buffer = deque(maxlen=100000)
gamma = 0.95

for episode in range(1000):
    state = env.reset()
    done = False
    while not done:
        # Sélection d'action avec bruit pour exploration (politique centrée sur la vente)
        state_t = torch.FloatTensor(state).unsqueeze(0)
        inv_t = torch.FloatTensor(env.inv).unsqueeze(0)
        with torch.no_grad():
            a = model.nash_action(state_t, inv_t).cpu().numpy()[0]
        noise = np.random.normal(0.0, 0.2, env.N)
        a = a + noise
        a = np.minimum(a, 0.0)  # on ne vend (négatif) que
        max_sell = -env.inv / env.dt
        a = np.maximum(a, max_sell)  # ne pas vendre plus que ce qu'on a
        a = np.clip(a, -2.0, 0.0)

        next_state, reward, done = env.step(a)
        buffer.append((state, a, reward, next_state, done, env.inv.copy()))

        state = next_state

        # Mise à jour si assez d'expérience
        if len(buffer) >= 64:
            batch = random.sample(buffer, 64)
            states, actions, rewards, states2, dones, invs = zip(*batch)
            S = torch.FloatTensor(states)
            A = torch.FloatTensor(actions)
            R = torch.FloatTensor(rewards) / 5.0  # normalisation
            S2 = torch.FloatTensor(states2)
            D = torch.FloatTensor(dones).unsqueeze(1)
            INV = torch.FloatTensor(invs)
            with torch.no_grad():
                A2 = target.nash_action(S2, INV)
                Q2, _ = target(S2, INV, A2)
                Y = R + gamma * Q2 * (1 - D)

            Q_pred, _ = model(S, INV, A)
            lossV = F.smooth_l1_loss(Q_pred, Y)
            optimizer_V.zero_grad()
            lossV.backward()
            optimizer_V.step()

            # Loss de régularisation sur psi (L1)
            _, psi = model(S, INV, A)
            lossA = 5.0 * psi.abs().mean()
            optimizer_A.zero_grad()
            lossA.backward()
            optimizer_A.step()

            # Soft update de la cible
            for tp, p in zip(target.parameters(), model.parameters()):
                tp.data.copy_(0.002*p.data + 0.998*tp.data)


/var/folders/hv/y00c8cl91z34xlwn9xbh5cfr0000gn/T/ipykernel_70251/4019850325.py:39: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:257.)
  S = torch.FloatTensor(states)


In [4]:
def rollout(env, policy):
    """
    Exécute un épisode complet avec la politique policy.
    Retourne les timestamps, les inventaires agents et le prix évoluant.
    """
    state = env.reset()
    invs, prices, timestamps = [], [], []
    for _ in range(env.M):
        invs.append(env.inv.copy())
        prices.append(env.S)
        timestamps.append(env.t)
        action = policy(state, env)
        state, _, done = env.step(action)
        if done: break
    return np.array(timestamps), np.array(invs), np.array(prices)

def nash_policy(state, env):
    """
    Politique Nash dérivée du réseau (NashDQN). Applique les mêmes contraintes de non-vente.
    """
    x = torch.FloatTensor(state).unsqueeze(0)
    inv = torch.FloatTensor(env.inv).unsqueeze(0)
    with torch.no_grad():
        a = model.nash_action(x, inv).cpu().numpy()[0]
    a = np.minimum(a, 0.0)
    max_sell = -env.inv / env.dt
    a = np.maximum(a, max_sell)
    return a

def twap_policy(state, env):
    """
    Baseline TWAP : vendre à taux constant pour liquider sur T.
    """
    rate = 1.0 / env.T  # on part de 1.0 inventaire initial
    return -np.ones(env.N) * rate

# Simulation d'exemple
env_nash = TradingEnv()
t_nash, inv_nash, price_nash = rollout(env_nash, nash_policy)
env_twap = TradingEnv()
t_twap, inv_twap, price_twap = rollout(env_twap, twap_policy)
